[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-04-model-validators-custom-types.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Model Validators and Custom Types — Cross-Field Logic
**certified-journeys / pydantic-certified** · Day 4 · Validation Deep Dive

> **Goal for today:** Write cross-field validation rules with `@model_validator`, build reusable custom types with `Annotated` + `BeforeValidator`, and create list-wrapping models with `RootModel`.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings


## Step 1 · `@model_validator(mode='after')` — Cross-Field Business Rules

Field validators run independently on each field. For rules that span *two or more* fields — like "end date must be after start date" — you need `@model_validator`.

**`mode='after'`** receives a **fully-constructed model instance**: all fields have already been parsed, coerced, and individually validated. You can read `self.field_a` and `self.field_b` safely.

| Hook | Receives | Use for |
|---|---|---|
| `@field_validator` | Single field value | Single-field coercion / validation |
| `@model_validator(mode='after')` | Full model instance | Cross-field rules |
| `@model_validator(mode='before')` | Raw input dict/object | Pre-processing before any parsing |

> **Reading:** [Pydantic Model Validators](https://docs.pydantic.dev/latest/concepts/validators/#model-validators)


In [ ]:
from datetime import date
from pydantic import BaseModel, model_validator
from pydantic import ValidationError


class DateRange(BaseModel):
    start_date: date
    end_date: date
    label: str = ""

    # mode='after' → self is a fully-validated DateRange instance
    @model_validator(mode="after")
    def check_end_after_start(self) -> "DateRange":
        if self.end_date <= self.start_date:
            raise ValueError(
                f"end_date ({self.end_date}) must be after start_date ({self.start_date})"
            )
        return self  # always return self from mode='after'


# Valid range
r = DateRange(start_date="2024-01-01", end_date="2024-03-31", label="Q1 2024")
print("Valid range:", r)
print("Duration days:", (r.end_date - r.start_date).days)

# Invalid: end before start
try:
    bad = DateRange(start_date="2024-06-01", end_date="2024-01-01")
except ValidationError as e:
    print("\nValidation error:")
    for err in e.errors():
        print(f"  loc={err['loc']}  msg={err['msg']}")

# Invalid: equal dates
try:
    same = DateRange(start_date="2024-04-15", end_date="2024-04-15")
except ValidationError as e:
    print("\nEqual-date error:")
    for err in e.errors():
        print(f"  msg={err['msg']}")


### What just happened?
- **`mode='after'`** fires after every field has been individually parsed — `self.start_date` and `self.end_date` are already `date` objects, not raw strings.
- **Returning `self`** is required; Pydantic uses the return value as the final model instance.
- The error lands on `__root__` / the model level (not on a specific field) because it's a cross-field rule — the `loc` tuple is empty.
- **Field validators cannot see other fields** — if you try to compare `end_date` inside a `@field_validator('end_date')`, `start_date` may not be validated yet.


## Step 2 · `@model_validator(mode='before')` — Pre-Processing Raw Input

`mode='before'` receives the **raw input** before Pydantic has parsed anything. The value is a `dict` (when called with keyword args or from JSON) or whatever was passed in.

Use cases:
- Rename legacy snake_case / camelCase keys to match your model
- Expand shorthand formats (`"2024-Q1"` → `{start_date: ..., end_date: ...}`)
- Strip unwanted fields before validation
- Inject computed defaults that depend on multiple input fields

| mode | Signature | Return type |
|---|---|---|
| `'before'` | `@classmethod` receiving raw data | Modified raw data (dict, etc.) |
| `'after'` | Instance method on validated model | `self` |


In [ ]:
from typing import Any
from pydantic import BaseModel, model_validator


class Event(BaseModel):
    name: str
    start_date: date
    end_date: date
    duration_days: int = 0  # will be filled by after-validator

    # mode='before' → cls method; data is the raw dict before any parsing
    @model_validator(mode="before")
    @classmethod
    def rename_legacy_keys(cls, data: Any) -> Any:
        if isinstance(data, dict):
            # Accept 'startDate' (camelCase) as an alias for 'start_date'
            if "startDate" in data and "start_date" not in data:
                data["start_date"] = data.pop("startDate")
            if "endDate" in data and "end_date" not in data:
                data["end_date"] = data.pop("endDate")
        return data  # return (possibly modified) raw data

    # mode='after' → instance method; self has fully-validated fields
    @model_validator(mode="after")
    def compute_duration(self) -> "Event":
        self.duration_days = (self.end_date - self.start_date).days
        return self


# Modern snake_case input
e1 = Event(name="PyCon", start_date="2024-05-17", end_date="2024-05-19")
print("Modern input:  ", e1)

# Legacy camelCase input from an old API
e2 = Event(**{"name": "DjangoCon", "startDate": "2024-09-22", "endDate": "2024-09-26"})
print("Legacy input:  ", e2)
print("Duration days: ", e2.duration_days)


### What just happened?
- **`mode='before'`** must be a `@classmethod` — no model instance exists yet; you only have raw data.
- The two validators run in sequence: `before` first (rename keys), then field parsing, then `after` (compute duration).
- **Mutating `data` in-place is safe** — Pydantic passes a copy of the dict in most cases, but returning the (modified) value makes the intent explicit.
- This pattern is the idiomatic way to support **multiple input formats** without duplicating your model definition.


## Step 3 · Custom Types with `Annotated` + `BeforeValidator`

Pydantic v2 lets you build **reusable custom types** using Python's `Annotated` — attach validators, metadata, and constraints directly to the type annotation.

```python
from typing import Annotated
from pydantic import BeforeValidator

MyType = Annotated[str, BeforeValidator(my_function)]
```

- **`BeforeValidator(fn)`** — runs `fn(value)` before Pydantic's own parsing
- **`AfterValidator(fn)`** — runs `fn(value)` after Pydantic's parsing
- Both accept a plain callable `(value: Any) -> Any`

This is far superior to subclassing: the type is **composable**, **importable**, and **JSON-schema-friendly**.

> **Reading:** [Pydantic Custom Types](https://docs.pydantic.dev/latest/concepts/types/#custom-types)


In [ ]:
import re
from typing import Annotated
from pydantic import BaseModel, BeforeValidator, ValidationError


def slugify(value: str) -> str:
    """Convert any string to a URL-safe slug."""
    value = str(value).strip().lower()
    # Replace spaces and underscores with hyphens
    value = re.sub(r"[\s_]+", "-", value)
    # Remove characters that aren't alphanumeric or hyphens
    value = re.sub(r"[^a-z0-9-]", "", value)
    # Collapse multiple consecutive hyphens
    value = re.sub(r"-+", "-", value)
    return value.strip("-")


# Annotated type: str, transformed by slugify before Pydantic validates it
Slug = Annotated[str, BeforeValidator(slugify)]


class BlogPost(BaseModel):
    title: str
    slug: Slug          # automatically normalised
    tags: list[Slug]    # works inside list too


post = BlogPost(
    title="Hello, World!",
    slug="  Hello, World!  ",      # raw messy string
    tags=["Python Tips", "Pydantic v2", "  type__safety  "],
)
print("slug: ", post.slug)   # → hello-world
print("tags: ", post.tags)   # → ['python-tips', 'pydantic-v2', 'type-safety']

# The Slug type is reusable in any model — no inheritance required
class Category(BaseModel):
    name: str
    slug: Slug

cat = Category(name="Data Engineering", slug="Data Engineering")
print("category slug:", cat.slug)  # → data-engineering


### What just happened?
- **`Slug = Annotated[str, BeforeValidator(slugify)]`** creates a type alias; any field typed `Slug` gets the slugify transform automatically.
- **`list[Slug]`** works — Pydantic applies the `BeforeValidator` to each element individually.
- The `Slug` type is defined once and reused across `BlogPost`, `Category`, and any future model — no duplication.
- **`BeforeValidator` vs `field_validator`**: prefer `BeforeValidator` for reusable transforms; use `@field_validator` for model-specific logic.


## Step 4 · `RootModel` — Validating Bare Lists and Scalars

Sometimes your API returns a **top-level array or scalar** — not an object with named fields. Pydantic's `RootModel` wraps these without needing an outer dict.

```python
from pydantic import RootModel

class IntList(RootModel[list[int]]):
    pass
```

Key differences from `BaseModel`:

| Feature | `BaseModel` | `RootModel` |
|---|---|---|
| Constructor | `Model(field=value)` | `Model(root=[1, 2, 3])` |
| Access data | `model.field` | `model.root` |
| JSON input | `{"field": ...}` | `[1, 2, 3]` (bare array) |
| Iteration | Not natively iterable | Wrap root to iterate |

> **Reading:** [Pydantic RootModel](https://docs.pydantic.dev/latest/concepts/models/#rootmodel-and-custom-root-types)


In [ ]:
import json
from typing import List
from pydantic import RootModel, ValidationError


# A model that validates a bare JSON array of integers
class IntList(RootModel[List[int]]):
    pass


# Construct from a Python list
nums = IntList(root=[1, 2, 3, 4, 5])
print("root:      ", nums.root)          # [1, 2, 3, 4, 5]
print("model_dump:", nums.model_dump())  # [1, 2, 3, 4, 5] — bare list, not a dict!

# Parse a raw JSON string that is a bare array
raw_json = "[10, 20, 30, 40]"
parsed = IntList.model_validate_json(raw_json)
print("from JSON: ", parsed.root)

# Serialise back to JSON
print("to JSON:   ", parsed.model_dump_json())  # '[10,20,30,40]'

# Coercion: strings that look like ints are coerced
coerced = IntList(root=["1", "2", "3"])  # '1' → 1
print("coerced:   ", coerced.root)

# Invalid: non-numeric string raises ValidationError
try:
    IntList(root=[1, "two", 3])
except ValidationError as e:
    print("\nValidation error:", e.errors()[0]["msg"])

# RootModel works with complex inner types too
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

class UserList(RootModel[List[User]]):
    pass

users_json = '[{"name": "Alice", "age": 30}, {"name": "Bob", "age": 25}]'
ul = UserList.model_validate_json(users_json)
print("\nUser list:")
for u in ul.root:
    print(f"  {u.name}, age {u.age}")


### What just happened?
- **`model_dump()`** on a `RootModel` returns the bare inner value (a list), not a `{"root": ...}` dict — this is intentional.
- **`model_validate_json`** parses a raw JSON string directly, skipping the intermediate Python parse step.
- `RootModel` composes with any Pydantic-supported type: `list[User]`, `dict[str, int]`, `Optional[str]`, etc.
- The `.root` attribute is the idiomatic way to access the wrapped value; you can also subclass and add methods that operate on `self.root`.


## Step 5 · Combining Everything — A Real-World Example

Real APIs combine all three patterns:
- `@model_validator(mode='before')` to normalise incoming keys
- `@model_validator(mode='after')` to enforce business invariants
- Custom `Annotated` types for reusable field transforms

Below we model a `BookingRequest` that could arrive from a web form or a legacy system.


In [ ]:
from typing import Annotated, Any
from datetime import date
from pydantic import BaseModel, BeforeValidator, ValidationError, model_validator


# Reusable type: strips and title-cases a name
def normalise_name(v: Any) -> str:
    return str(v).strip().title()

NormalisedName = Annotated[str, BeforeValidator(normalise_name)]

# Reusable type: parses flexible date strings
def parse_date_flexible(v: Any) -> date:
    if isinstance(v, date):
        return v
    s = str(v).strip()
    # Support YYYY/MM/DD in addition to YYYY-MM-DD
    s = s.replace("/", "-")
    return date.fromisoformat(s)

FlexDate = Annotated[date, BeforeValidator(parse_date_flexible)]


class BookingRequest(BaseModel):
    guest_name: NormalisedName
    check_in: FlexDate
    check_out: FlexDate
    room_count: int = 1
    nights: int = 0  # computed

    @model_validator(mode="before")
    @classmethod
    def handle_legacy_keys(cls, data: Any) -> Any:
        """Accept legacy camelCase keys from old booking systems."""
        if isinstance(data, dict):
            mapping = {
                "guestName": "guest_name",
                "checkIn": "check_in",
                "checkOut": "check_out",
                "roomCount": "room_count",
            }
            for old_key, new_key in mapping.items():
                if old_key in data and new_key not in data:
                    data[new_key] = data.pop(old_key)
        return data

    @model_validator(mode="after")
    def validate_stay(self) -> "BookingRequest":
        if self.check_out <= self.check_in:
            raise ValueError("check_out must be after check_in")
        if self.room_count < 1:
            raise ValueError("room_count must be at least 1")
        self.nights = (self.check_out - self.check_in).days
        return self


# Modern input
b1 = BookingRequest(
    guest_name="  alice smith  ",
    check_in="2024/07/15",
    check_out="2024/07/18",
    room_count=2,
)
print("Modern input:")
print(f"  guest: {b1.guest_name}, check_in: {b1.check_in}, nights: {b1.nights}")

# Legacy camelCase input
b2 = BookingRequest(**{
    "guestName": "BOB JONES",
    "checkIn": "2024-08-01",
    "checkOut": "2024-08-05",
})
print("\nLegacy input:")
print(f"  guest: {b2.guest_name}, nights: {b2.nights}")

# Invalid: check-out before check-in
try:
    BookingRequest(guest_name="Carol", check_in="2024-09-10", check_out="2024-09-05")
except ValidationError as e:
    print("\nError:", e.errors()[0]["msg"])


### What just happened?
- Three concerns are cleanly separated: key normalisation (`before`), field transforms (`Annotated` types), business rules (`after`).
- **`FlexDate`** accepts both `YYYY-MM-DD` and `YYYY/MM/DD` transparently — the `BeforeValidator` converts before Pydantic's date parser runs.
- **`NormalisedName`** is reusable across any model that needs name normalisation — zero duplication.
- The `after` validator both validates (check_out > check_in) and computes a derived field (nights) in a single pass.


In [ ]:
# Challenge: Custom type + RootModel
#
# 1. Create a custom Annotated type `PositiveInt` that uses a BeforeValidator
#    to convert floats/strings to int, then an AfterValidator to raise ValueError
#    if the result is <= 0.
#
# 2. Create a model `Product` with fields:
#      name: str
#      price: float  (must be > 0)
#      stock: PositiveInt
#
# 3. Add a @model_validator(mode='after') that sets a `low_stock` bool field
#    to True when stock < 10.
#
# 4. Create a `ProductCatalog = RootModel[list[Product]]` and parse:
#    '[{"name": "Widget", "price": 9.99, "stock": "5"},
#      {"name": "Gadget", "price": 24.99, "stock": "50"}]'
#
# Expected: Widget has low_stock=True, Gadget has low_stock=False

from typing import Annotated
from pydantic import BaseModel, BeforeValidator, AfterValidator, RootModel, model_validator

# Your solution here
# def to_int(v): ...
# def must_be_positive(v): ...
# PositiveInt = Annotated[int, ...]

# class Product(BaseModel):
#     ...

# class ProductCatalog(RootModel[list[Product]]):
#     pass


---
## Day 4 key concepts recap

| Concept | What to remember |
|---|---|
| `@model_validator(mode='after')` | Receives fully-validated model instance; use for cross-field rules |
| `@model_validator(mode='before')` | `@classmethod`; receives raw input; use for key renaming / pre-processing |
| `Annotated` + `BeforeValidator` | Reusable type transforms — apply to any field without subclassing |
| `AfterValidator` | Runs after Pydantic's own parsing; receives the already-typed value |
| `RootModel[T]` | Wraps bare JSON arrays/scalars; `.root` holds the inner value |
| Execution order | `mode='before'` → field parsing → field validators → `mode='after'` |

> **Tip:** `@model_validator(mode='after')` receives a fully-validated model instance — all field validators have already run. This makes it the right place for cross-field business rules, not pre-parsing transformations.

---
## What's next
**Day 5** → Serialization: `model_dump`, `model_dump_json`, `@computed_field`, `@field_serializer`, and JSON Schema generation.

Mark Day 4 complete in your [tracker](../index.html).
